In [5]:
# import modules
import os
import pandas as pd
import numpy as np


In [6]:
# call the" already detrend data for the stations 
datadir = "/Users/lhylu/Library/CloudStorage/OneDrive-Personal/CODING/GNSS.processing/POS-FILES_1days-sol"
base_dir = '/Users/lhylu/Library/CloudStorage/OneDrive-Personal/CODING/GNSS.processing/results/'
fit_dir = os.path.join(base_dir, 'fit_csv')
out_dir = os.path.join(base_dir, 'rates')

os.makedirs(out_dir, exist_ok=True) # create output directory if not exists

## get position from original data file
def get_coords(pos_data):
    """
    Open the .ovs.final_igb14.pos file and read coordinates from the line 
    starting with 'NEU Reference position'. 
    Returns (lat, lon, height) as floats, or np.nan if not found.
    """
    try:
        with open(pos_data, "r") as f:
            for line in f:
                if line.strip().startswith("NEU Reference position"):
                    words = line.split()  # split by any whitespace
                    # Example line:
                    # NEU Reference position : <lat> <lon> <height>
                    lat = float(words[4])
                    lon = float(words[5])
                    if lon > 180.0:       # normalize longitude to [-180,180)
                        lon -= 360.0
                    height = float(words[6])
                    return lat, lon, height
    except Exception as e:
        print(f"[WARN] Could not read coordinates from {pos_data}: {e}")
    # If nothing found, return NaN values (safer for formatting later)
    return np.nan, np.nan, np.nan

# check the files in the fit directory
csv_files = sorted([f for f in os.listdir(fit_dir) if f.endswith('.csv')])
print(f"Found {len(csv_files)} CSV files in {fit_dir}")
print("Calculating linear rates for each station...")


# create a map of station to coordinates from the original data files
stations = sorted({fn.split("_")[0] for fn in csv_files})
coords_map = {}
for stat in stations:
    pos_path = os.path.join(datadir, f"{stat}.ovs.final_igb14.pos")
    lat, lon, height = get_coords(pos_path)
    coords_map[stat] = (lat, lon, height)

# prepare the output rates file
out_path = os.path.join(out_dir, 'gnss_detrended_rates.txt')

# heading for the output rates file
with open(out_path, 'w') as f:
    f.write('# STAT     Lat      Long   Height "Npos_fitted" "Epos_fitted" "Upos_fitted"    N_fit_err     E_fit_err     U_fit_err    Sdate       Edate\n')
    f.write('#           °        °      m          mm/yr    mm/yr    mm/yr    mm/yr    mm/yr     mm/yr                          YEAR-MO-DY  YEAR-MO-DY\n')
    f.write('#-----------------------------------------------------------------------------------------------------------------------------------------\n')


# loop through each csv file and calculate rates
for csv_name in csv_files:
    try:
        stat = csv_name.split("_")[0]  # station name = first part of filename

        # Load detrended time series
        df = pd.read_csv(os.path.join(fit_dir, csv_name))

        # Ensure Date column is parsed correctly
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df = df.sort_values("Date")
        sdate = df["Date"].iloc[0]
        edate = df["Date"].iloc[-1]
        sdate_str = sdate.strftime("%Y-%m-%d")
        edate_str = edate.strftime("%Y-%m-%d")

        # Time axis (decimal years)
        x = pd.to_numeric(df["stat_decimal_year"], errors="coerce").to_numpy()

        # Fitted displacements (already in mm)
        yN = pd.to_numeric(df["Npos_fitted"], errors="coerce").to_numpy()
        yE = pd.to_numeric(df["Epos_fitted"], errors="coerce").to_numpy()
        yU = pd.to_numeric(df["Upos_fitted"], errors="coerce").to_numpy()

        # Calculate slopes = rates in mm/yr
        mN_slope = np.polyfit(x, yN, 1)[0]
        mE_slope = np.polyfit(x, yE, 1)[0]
        mU_slope = np.polyfit(x, yU, 1)[0]

        # Placeholder uncertainties (can be calculated later)
        N_fit_err = E_fit_err = U_fit_err = float("nan")

        # Get coordinates from original .pos file (or NaN if missing)
        lat, lon, height = coords_map.get(stat, (np.nan, np.nan, np.nan))

        # Write results to output file
        with open(out_path, "a") as ratesFile:
            ratesFile.write(
                f"{stat:<6s}  {lat:8.4f} {lon:9.4f} {height:9.3f}  "
                f"{mN_slope:9.3f} {mE_slope:9.3f} {mU_slope:9.3f}  "
                f"{N_fit_err:10.3f} {E_fit_err:10.3f} {U_fit_err:10.3f}  "
                f"{sdate_str}  {edate_str}\n"
            )
        print(f"Processed {stat}: N={mN_slope:.3f}, E={mE_slope:.3f}, U={mU_slope:.3f} mm/yr")

    except Exception as e:
        print(f"[WARN] file error {csv_name}: {e}")

print(f"[OK] Rates written to {out_path}")



Found 49 CSV files in /Users/lhylu/Library/CloudStorage/OneDrive-Personal/CODING/GNSS.processing/results/fit_csv
Calculating linear rates for each station...
Processed ABEJ: N=0.018, E=0.002, U=0.001 mm/yr
Processed AROL: N=0.014, E=0.000, U=0.004 mm/yr
Processed BIJA: N=0.007, E=-0.003, U=-0.000 mm/yr
Processed BON2: N=0.021, E=-0.002, U=-0.009 mm/yr
Processed BRBR: N=0.010, E=0.003, U=-0.004 mm/yr
Processed CABA: N=0.019, E=-0.002, U=0.006 mm/yr
Processed CAPO: N=0.014, E=0.007, U=-0.008 mm/yr
Processed CDME: N=0.019, E=0.005, U=0.001 mm/yr
Processed CDTO: N=0.018, E=0.005, U=-0.000 mm/yr
Processed CHLS: N=0.009, E=-0.001, U=0.002 mm/yr
Processed CHOM: N=0.018, E=-0.001, U=0.001 mm/yr
Processed COBB: N=0.016, E=-0.007, U=-0.006 mm/yr
Processed COVE: N=0.007, E=-0.001, U=0.000 mm/yr
Processed CRU2: N=0.015, E=-0.007, U=0.002 mm/yr
Processed CTCR: N=0.014, E=0.005, U=0.001 mm/yr
Processed EART: N=0.010, E=0.001, U=-0.001 mm/yr
Processed ELVI: N=0.019, E=-0.004, U=0.006 mm/yr
Processed 